# Tool Use 기초 (1) — 도구 함수와 스키마

**Skilljar Lessons 03-04 대응**

이 노트북에서 다루는 내용:
1. 도구 함수(Tool Function) 작성
2. 도구 스키마(Tool Schema) 정의
3. 첫 번째 Tool Use API 호출

In [1]:
# ── Setup ──────────────────────────────────────────────
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. 도구 함수 작성 (Tool Functions)

Claude가 **직접 실행하지 않는**, 우리가 작성하는 일반 Python 함수입니다.  
Claude는 이 함수를 호출해 달라고 **요청**만 하고, 실제 실행은 우리 코드가 합니다.

Reminder System의 첫 번째 도구: **현재 날짜/시간 조회**

In [2]:
from datetime import datetime


def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    현재 날짜와 시간을 지정된 포맷으로 반환합니다.

    Args:
        date_format: strftime 포맷 문자열 (기본값: "%Y-%m-%d %H:%M:%S")

    Returns:
        포맷된 날짜/시간 문자열
    """
    valid_formats = [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d",
        "%H:%M:%S",
        "%H:%M",
        "%Y/%m/%d",
        "%m/%d/%Y",
    ]
    if date_format not in valid_formats:
        raise ValueError(
            f"Invalid date format: {date_format}. "
            f"Valid formats: {valid_formats}"
        )
    return datetime.now().strftime(date_format)

In [3]:
# 함수 테스트
print("기본 포맷:", get_current_datetime())
print("시:분 포맷:", get_current_datetime("%H:%M"))

기본 포맷: 2026-04-14 13:26:22
시:분 포맷: 13:26


## §2. 도구 스키마 정의 (Tool Schemas)

Claude에게 도구의 **이름**, **설명**, **입력 파라미터**를 알려주는 JSON Schema입니다.  
Claude는 이 스키마를 읽고 언제·어떻게 도구를 사용할지 판단합니다.

스키마의 3가지 핵심 필드:
- `name` — 도구 이름 (영문, snake_case)
- `description` — 도구가 하는 일 (Claude가 읽는 설명)
- `input_schema` — JSON Schema 형식의 파라미터 정의

In [4]:
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time in a specified format. "
        "Defaults to '%Y-%m-%d %H:%M:%S'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "The format string for the date/time output. "
                    "Supported formats: "
                    "'%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%H:%M:%S', "
                    "'%H:%M', '%Y/%m/%d', '%m/%d/%Y'."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

print("Schema name:", get_current_datetime_schema["name"])
print("Properties:", list(get_current_datetime_schema["input_schema"]["properties"].keys()))

Schema name: get_current_datetime
Properties: ['date_format']


In [6]:
from anthropic.types import ToolParam

get_current_datetime_schema: ToolParam = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time in the specified format. "
                   "Use this tool when you need to know the current date or time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "The format string for the date/time output. "
                              "Uses Python strftime format codes. "
                              "Default: '%Y-%m-%d %H:%M:%S'",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}

print("Schema name:", get_current_datetime_schema["name"])
print("Properties:", list(get_current_datetime_schema["input_schema"]["properties"].keys()))

Schema name: get_current_datetime
Properties: ['date_format']


## §3. 첫 Tool Use 호출 (First Tool Use Call)

`tools` 파라미터에 스키마를 전달하면, Claude는 필요할 때 도구 사용을 **요청**합니다.  
- `stop_reason`이 `"tool_use"`이면 → Claude가 도구 호출을 원한다는 뜻
- `response.content`에 `ToolUseBlock`이 포함됩니다

In [12]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[get_current_datetime_schema],
    messages=[
        {"role": "user", "content": "What time is it right now?"}
    ],
)

print("Response:", response)
print("stop_reason:", response.stop_reason)
print()
for block in response.content:
    print(f"  type: {block.type}")
    if block.type == "tool_use":
        print(f"  name: {block.name}")
        print(f"  input: {block.input}")
        print(f"  id:   {block.id}")

Response: Message(id='msg_01EgemGE7jfNX781LUWhNkn1', container=None, content=[ToolUseBlock(id='toolu_01E1k1mA9e9De2wH5rM6ugNs', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=637, output_tokens=39, server_tool_use=None, service_tier='standard'), stop_details=None)
stop_reason: tool_use

  type: tool_use
  name: get_current_datetime
  input: {}
  id:   toolu_01E1k1mA9e9De2wH5rM6ugNs


In [6]:
response

Message(id='msg_01HP6EhwHPe8D6T6PNEKKW5r', container=None, content=[ToolUseBlock(id='toolu_0115FVb2t9PMRykonJbJm2pD', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=668, output_tokens=39, server_tool_use=None, service_tier='standard'))

In [7]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What time is it right now?"}
    ],
)

In [8]:
response

Message(id='msg_01Eh9a7oNeQVwXpPA8gYKdtL', container=None, content=[TextBlock(citations=None, text='I don\'t have access to real-time information, so I can\'t tell you the current time. \n\nTo find out what time it is, you can:\n- Check your phone, computer, or watch\n- Ask a voice assistant like Siri, Alexa, or Google Assistant\n- Search "current time" online\n\nIs there anything else I can help you with?', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=14, output_tokens=84, server_tool_use=None, service_tier='standard'))